# LAB 3 — HyDE: Hypothetical Document Embeddings
## Aula 6: Indexação Avançada · MBA RAG & CAG Aplicados a Direito e Segurança Pública

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

**Objetivo:** Implementar HyDE para reduzir o gap semântico entre queries coloquiais e linguagem técnico-jurídica, visualizando a melhoria geometricamente.

**Tempo estimado:** 40 minutos | **Pré-requisito:** Labs 1 e 2 concluídos

## ⚙️ Passo 1 — Instalação

In [ ]:
!pip install sentence-transformers langchain langchain-openai faiss-cpu umap-learn matplotlib --quiet
print('✅ OK')

## 📦 Passo 2 — Imports

In [ ]:
import json, os, warnings
import numpy as np
import matplotlib.pyplot as plt
import faiss
import umap
from sentence_transformers import SentenceTransformer
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
warnings.filterwarnings('ignore')

try:
    encoder = SentenceTransformer('BAAI/bge-m3')
    print('BGE-M3 carregado')
except:
    encoder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
    print('Fallback: MiniLM')

try:
    llm = ChatOpenAI(model='meta-llama/Llama-3.1-8B-Instruct', base_url='http://localhost:8000/v1', api_key='dummy', temperature=0.3, max_tokens=200)
    llm.invoke('ok')
    LLM_ON = True
    print('vLLM conectado')
except:
    LLM_ON = False
    print('vLLM offline')

## 📚 Passo 3 — Corpus e Índice FAISS

In [ ]:
corpus_docs = [
    {'id':'DOC001','texto':'O STF determinou no HC 143641 a substituicao da prisao preventiva por domiciliar para gestantes e maes de criancas ate 12 anos conforme art 318 CPP com efeitos erga omnes.'},
    {'id':'DOC002','texto':'O Pacote Anticrime Lei 13964/2019 art 158-A define cadeia de custodia como conjunto de procedimentos para documentar historia cronologica do vestigio coletado. Art 3-B cria juiz das garantias.'},
    {'id':'DOC003','texto':'O compartilhamento de relatorios do COAF com MP sem autorizacao judicial e constitucional conforme Tema 990 STF RE 1055941. Dados devem ter sido obtidos no exercicio regular das atribuicoes.'},
    {'id':'DOC006','texto':'STJ decidiu RHC 143308 que busca pessoal sem mandado exige fundada suspeita objetiva concreta conforme art 240 paragrafo 2 CPP. Nervosismo e local de trafico nao sao suficientes.'},
    {'id':'DOC005','texto':'Laudo pericial INC 2024 identificou 847 transacoes Bitcoin 12.4 BTC rastreado via Chainalysis Reactor para exchanges nao regulamentadas. Cadeia de custodia garantida por hash SHA-256.'},
    {'id':'DOC009','texto':'PGR concluiu que sistemas de reconhecimento facial com vies algoritmico nao podem ser prova direta penal. Servem apenas como elemento investigativo sujeito a confirmacao humana.'},
]

corpus_texts = [d['texto'] for d in corpus_docs]
corpus_embs = encoder.encode(corpus_texts, normalize_embeddings=True)
dim = corpus_embs.shape[1]

faiss_idx = faiss.IndexFlatIP(dim)
faiss_idx.add(corpus_embs.astype('float32'))
print(f'FAISS: {faiss_idx.ntotal} vetores de {dim}d')

## 🧠 Passo 4 — Pipeline HyDE

In [ ]:
HIPOTETIVOS = {
    'como provar lavagem com bitcoin?': 'Para demonstrar lavagem de ativos em criptomoedas o laudo pericial deve conter analise de blockchain certificada pelo INC correlacao temporal entre transacoes suspeitas e crimes antecedentes e identificacao dos enderecos de destino conforme Lei 9613/1998.',
    'mae presa pode ficar em casa?': 'Nos termos do HC 143641 STF e possivel substituicao da prisao preventiva por domiciliar para mulheres gestantes puerp. ou maes de criancas ate 12 anos conforme art 318 incisos IV V CPP ressalvadas hipoteses excepcionais.',
    'policial pode revistar na rua?': 'A busca pessoal sem mandado judicial condiciona-se nos termos art 240 paragrafo 2 CPP a fundada suspeita objetiva concreta de que o individuo oculte arma proibida ou objeto ilicito. Atitudes genericas nao satisfazem o requisito legal.',
}

HYDE_PROMPT = ChatPromptTemplate.from_messages([
    ('system','Escreva 80-120 palavras de um documento juridico brasileiro que responderia a pergunta. Use linguagem tecnica formal. Nao diga que e hipotetico.'),
    ('human','Pergunta: {question}\nDocumento hipotetico:'),
])

def hyde_search(query, top_k=3):
    # Busca direta
    q_e = encoder.encode([query], normalize_embeddings=True)[0]
    D_d, I_d = faiss_idx.search(q_e.reshape(1,-1).astype('float32'), top_k)

    # Gerar hipotetico
    if LLM_ON:
        chain = HYDE_PROMPT | llm | StrOutputParser()
        hipotetivo = chain.invoke({'question': query})
    else:
        hipotetivo = HIPOTETIVOS.get(query.lower(), query + ' legislacao brasileira juridico formal')

    # Busca com hipotetico
    h_e = encoder.encode([hipotetivo], normalize_embeddings=True)[0]
    D_h, I_h = faiss_idx.search(h_e.reshape(1,-1).astype('float32'), top_k)

    return {'query':query,'hipotetivo':hipotetivo,'q_emb':q_e,'h_emb':h_e,
            'direto':[(D_d[0][i],corpus_docs[I_d[0][i]]) for i in range(top_k)],
            'hyde':[(D_h[0][i],corpus_docs[I_h[0][i]]) for i in range(top_k)]}

print('Pipeline HyDE pronto')

## 🔬 Passo 5 — Comparação HyDE vs Busca Direta

In [ ]:
queries_test = [
    'como provar lavagem com bitcoin?',
    'mae presa pode ficar em casa?',
    'policial pode revistar na rua?',
]

resultados_hyde = []
print('='*60)
print('COMPARACAO: Busca Direta vs HyDE')
print('='*60)

for q in queries_test:
    r = hyde_search(q)
    resultados_hyde.append(r)

    d_score = np.mean([s for s,_ in r['direto']])
    h_score = np.mean([s for s,_ in r['hyde']])

    print(f'\nQuery: {r["query"]}')
    print(f'  Hipotetico: {r["hipotetivo"][:120]}...')
    print(f'  Direto top1: score={r["direto"][0][0]:.3f} | {r["direto"][0][1]["id"]}')
    print(f'  HyDE top1:   score={r["hyde"][0][0]:.3f} | {r["hyde"][0][1]["id"]}')
    print(f'  Delta: {h_score-d_score:+.3f}')

melhoria_media = np.mean([
    np.mean([s for s,_ in r['hyde']]) - np.mean([s for s,_ in r['direto']])
    for r in resultados_hyde
])
print(f'\nMelhoria media de score: {melhoria_media:+.4f}')

## 📊 Passo 6 — Visualização Geométrica do Gap Semântico

In [ ]:
# Reduzir todos os embeddings juntos para 2D
n_nb = min(5, len(corpus_docs) + len(queries_test) * 2 - 1)
all_to_reduce = np.vstack([
    corpus_embs,
    np.array([r['q_emb'] for r in resultados_hyde]),
    np.array([r['h_emb'] for r in resultados_hyde]),
])
reducer = umap.UMAP(n_components=2, n_neighbors=n_nb, metric='cosine', random_state=42)
all_2d = reducer.fit_transform(all_to_reduce)

nc = len(corpus_docs)
nq = len(queries_test)
corpus_2d = all_2d[:nc]
queries_2d = all_2d[nc:nc+nq]
hyps_2d = all_2d[nc+nq:]

fig, ax = plt.subplots(figsize=(10,7))

ax.scatter(corpus_2d[:,0], corpus_2d[:,1], c='steelblue', s=200, label='Corpus (docs reais)', zorder=3, marker='s', edgecolors='w', lw=1.5)
for i, d in enumerate(corpus_docs):
    ax.annotate(d['id'], corpus_2d[i], fontsize=7, color='steelblue', xytext=(0,6), textcoords='offset points')

ax.scatter(queries_2d[:,0], queries_2d[:,1], c='tomato', s=150, label='Queries coloquiais', zorder=4, marker='^', edgecolors='w', lw=1.5)
ax.scatter(hyps_2d[:,0], hyps_2d[:,1], c='mediumseagreen', s=150, label='Docs hipoteticos (HyDE)', zorder=4, marker='D', edgecolors='w', lw=1.5)

# Setas query -> hipotetico
for i in range(nq):
    ax.annotate('',xytext=queries_2d[i],xy=hyps_2d[i],arrowprops=dict(arrowstyle='->',color='orange',lw=2))
    ax.annotate(f'Q{i+1}', queries_2d[i], fontsize=8, color='tomato', xytext=(4,4), textcoords='offset points')
    ax.annotate(f'H{i+1}', hyps_2d[i], fontsize=8, color='mediumseagreen', xytext=(4,4), textcoords='offset points')

ax.set_title('Gap Semantico: Query Coloquial -> Hipotetico -> Corpus\nSetas laranja = traducao semantica do HyDE', fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('hyde_gap_semantico.png', dpi=120)
plt.show()
print('Salvo: hyde_gap_semantico.png')
print('Observe como os hipoteticos (verde) ficam mais proximos ao corpus do que as queries (vermelho)')

## ⚠️ Passo 7 — Quando HyDE Falha

In [ ]:
print('DIAGNOSTICO: Casos onde HyDE pode ser contraproducente')
print('='*60)
casos = [
    ('Queries com entidades especificas', 'qual e o numero do HC sobre maes presas?', 'LLM pode alucinar numero diferente. Use BM25.'),
    ('Corpus pequeno (<50 docs)', 'qualquer query', f'Corpus com {len(corpus_docs)} docs. Ganho marginal. Nao compensa latencia.'),
    ('Queries ja tecnicas', 'fundada suspeita art 240 paragrafo 2 CPP', 'Hipotetico quase identico a query. Custo sem ganho.'),
]
for tipo, q, rec in casos:
    print(f'\n❌ {tipo}')
    print(f'   Exemplo: "{q}"')
    print(f'   Recomendacao: {rec}')

print('\n✅ USE HyDE quando: usuarios leigos + corpus extenso (>100 docs) + gap semantico confirmado')

## 💾 Passo 8 — Salvar Resultados

In [ ]:
import os
os.makedirs('resultados_aula6', exist_ok=True)

metricas = {
    'tecnica': 'HyDE',
    'queries_testadas': [r['query'] for r in resultados_hyde],
    'melhoria_media_score': float(melhoria_media),
    'hipoteticos': [r['hipotetivo'][:150] for r in resultados_hyde],
}

with open('resultados_aula6/resultados_hyde.json','w',encoding='utf-8') as f:
    json.dump(metricas, f, ensure_ascii=False, indent=2)

print('✅ Salvo: resultados_aula6/resultados_hyde.json')
print(f'Melhoria media: {melhoria_media:+.4f}')

## 📚 Referências (ABNT)

GAO, L. et al. **Precise Zero-Shot Dense Retrieval without Relevance Labels**. In: *ACL*, Toronto, 2023. arXiv:2212.10496.

---
*MBA em RAG & CAG Aplicados a Direito e Segurança Pública — Aula 6, Lab 3*